## Machine Learning Model

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import root_mean_squared_error, mean_absolute_error,r2_score

from xgboost import XGBRegressor

In [21]:
#### Load cleaned data
df = pd.read_csv(r"C:\Users\ayush\OneDrive\Desktop\AirBNB\cleaned_airbnb_data.csv")

In [22]:
df.head()

,room_id,name,host_id,neighbourhood,room_type,room_price,minimum_nights,number_of_reviews,last_review_date,reviews_per_month,rooms_by_host,availability,city,country,location,latitude,longitude
0,9153958,.2 La Casa sui Tetti dell'Oltrarno,47655975,Centro Storico,Entire home/apt,50.0,1,363,363,6.69,2,0,Florence,Italy,"Italy, Florence, Centro Storico",43.766789,11.245890
1,9191572,Firenze Studio Porta Romana,47816872,Centro Storico,Entire home/apt,70.0,2,46,46,0.83,1,364,Florence,Italy,"Italy, Florence, Centro Storico",43.765447,11.242550
2,9306481,ELEGANT FLAT FLORENCE CITY CENTER,42112573,Centro Storico,Entire home/apt,120.0,2,132,132,2.42,1,270,Florence,Italy,"Italy, Florence, Centro Storico",43.779967,11.261314
3,9524225,Luxury Modern Studio - Florence,11065786,Campo di Marte,Entire home/apt,75.0,3,56,56,1.01,5,364,Florence,Italy,"Italy, Florence, Campo di Marte",43.768156,11.275231
4,9602552,Florence is a dream,956926,Campo di Marte,Entire home/apt,50.0,2,24,24,0.44,1,365,Florence,Italy,"Italy, Florence, Campo di Marte",43.788093,11.273348


In [24]:
#### Encode the categorical features
le_room = LabelEncoder()
le_city = LabelEncoder()

df['room_type_encoded'] = le_room.fit_transform(df['room_type'])
df['city_encoded'] = le_city.fit_transform(df['city'])

#### Feature engineering 
df['price_per_night']= df['room_price']/ df['minimum_nights'].replace(0,1)
df['review_intensity'] = df['number_of_reviews']/ (df['reviews_per_month'].replace(0,1))

In [25]:
#### Select the features and target
features = ['room_type_encoded', 'city_encoded', 'minimum_nights', 'number_of_reviews', 
        'reviews_per_month', 'availability','price_per_night', 'review_intensity']
x = df[features]
y = df['room_price']

In [26]:
#### Split the data into train and test 
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=0.2,random_state=42)

In [29]:
#### Build the xgboost regresssor model and hyperparameters
model = XGBRegressor(random_state=42)

param_grid = {
    'max_depth' : [3,5,7],
    'n_estimators' : [100,200],
    'learning_rate' : [0.01, 0.1, 0.2],
    'subsample': [0.7, 1]       
}
grid_search = GridSearchCV(estimator=model, param_grid=param_grid,cv=3,

scoring='neg_mean_absolute_error', verbose=2, n_jobs=-1)

grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_

Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [17]:
#### Predict on test data 
y_pred =best_model.predict(X_test)

#### Fix negative prices
y_pred = np.maximum(y_pred,0)


In [30]:
#### Evaluation
rmse = root_mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# R2 Score
r2 = r2_score(y_test, y_pred)

print(f"Final RMSE: {rmse:.2f}")
print(f"Final MAE: {mae:.2f}")
print(f"Final R2 Score: {r2:.2f}")

Final RMSE: 503964.93
Final MAE: 2654.32
Final R2 Score: -0.00


In [37]:
 # Predict on full dataset
full_predictions = best_model.predict(X)
full_predictions = np.maximum(full_predictions, 0)  # fix negatives if needed

# Drop columns that is not needed
df.drop(['room_type_encoded', 'city_encoded'], axis=1, inplace=True)

# Add predicted prices to original data
df['Predicted Price'] = full_predictions

# Optional: Rename for consistency
df.rename(columns={'room_price': 'Actual Price'}, inplace=True)

# Save final dataset for Power BI
df.to_csv('airbnb_dynamic_pricing_final.csv', index=False)
print("Saved full data with predictions for Power BI.")

Saved full data with predictions for Power BI.
